# Weather Agent — Bug Fix Documentation

This notebook documents all bugs found and fixed in the Weather Agent codebase. The agent had **9 bugs across 5 files** that prevented it from running correctly.

---
## Summary Table

| # | File | Bug | Fix |
|---|------|-----|-----|
| 1 | `nodes.py` | `location_data` set to `{}` instead of API response | Assign `location_data` variable |
| 2 | `nodes.py` | `wind_unit` commented out, used as string literal | Uncomment variable, use `{wind_unit}` in f-string |
| 3 | `graph.py` | `fetch_weather_data` node missing from edge sequence | Add edges: `fetch_location_data` → `fetch_weather_data` → `generate_weather_info` |
| 4 | `graph.py` | Graph never compiled | Change `builder` to `builder.compile()` |
| 5 | `main.py` | Greeting uses plain string, runs before name guard | Use f-string, move print after guard |
| 6 | `main.py` | `"_main_"` typo in entry point check | Change to `"__main__"` |
| 7 | `config.py` | `WEATHER_API_BASE_URL` declared twice, second is `""` | Remove duplicate empty declaration |
| 8 | `config.py` | `TEMP_MIN` is `"0"` string, not a readable label | Change value to `"freezing"` |
| 9 | `helper_functions.py` | `if temp_celsius:` catches all non-zero temps | Change condition to `if temp_celsius < 0:` |

---
## `nodes.py` — 2 Bugs
### Bug 1: Location data discarded after fetch

**Root cause:** After successfully fetching and validating the location API response, the result was thrown away and replaced with an empty dict.

**Impact:** `location_data` was always empty, causing all downstream nodes to fail with *"Location or weather data not available."*
# BEFORE (broken)
state["location_data"] = {}
# AFTER (fixed)
state["location_data"] = location_data
### Bug 2: Wind unit rendered as literal string

**Root cause:** The `wind_unit` variable was commented out, and the f-string used the variable name as a hardcoded string instead of interpolating it.

**Impact:** Wind output displayed literally as `4.0 wind_unit` instead of `4.0 km/h`.
# BEFORE (broken)
# wind_unit = units.get("windspeed", "km/h")   <-- commented out
weather_info_parts = [
    f"• Wind: {windspeed} wind_unit"            # string literal, not interpolated
]
# AFTER (fixed)
wind_unit = units.get("windspeed", "km/h")     # uncommented
weather_info_parts = [
    f"• Wind: {windspeed} {wind_unit}"          # correctly interpolated
]
---
## `graph.py` — 2 Bugs
### Bug 3: `fetch_weather_data` node missing from edge connections

**Root cause:** The edge sequence skipped from `fetch_location_data` directly to `generate_weather_info`, bypassing the weather fetch step entirely.

**Impact:** `weather_data` was never populated, so `generate_weather_info` always raised an exception.
# BEFORE (broken)
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "generate_weather_info")  # skips weather fetch!
builder.add_edge("generate_weather_info", END)
# AFTER (fixed)
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data")     # added missing edge
builder.add_edge("fetch_weather_data", "generate_weather_info")   # added missing edge
builder.add_edge("generate_weather_info", END)
### Bug 4: Graph never compiled

**Root cause:** The `builder` object was assigned directly to `weather_agent` without calling `.compile()`. LangGraph requires compilation to produce a runnable graph.

**Impact:** Calling `weather_agent.invoke(state)` would fail since `builder` has no `invoke` method.
# BEFORE (broken)
weather_agent = builder
# AFTER (fixed)
weather_agent = builder.compile()
---
## `main.py` — 2 Bugs
### Bug 5: Greeting printed before name validation, using wrong string format

**Root cause:** The greeting was printed before the empty-name guard, and used a regular string instead of an f-string, so `{name}` was printed literally.

**Impact:** Would print `Hello {name}!` literally, or `Hello !` for empty input.
# BEFORE (broken)
name = input("Enter your name: ").strip()
print("Hello {name}!")          # plain string, not f-string; runs before guard
if not name:
    name = "User"
# AFTER (fixed)
name = input("Enter your name: ").strip()
if not name:
    name = "User"
print(f"Hello {name}!")         # f-string, runs after guard
### Bug 6: `__main__` entry point check had typo

**Root cause:** Single underscores were used instead of the required double underscores.

**Impact:** Running the script directly (`python main.py`) would never call `main()`, so the program would silently do nothing.
# BEFORE (broken)
if __name__ == "_main_":    # single underscores — never matches
    main()
# AFTER (fixed)
if __name__ == "__main__":  # double underscores
    main()
---
## `config.py` — 2 Bugs
### Bug 7: `WEATHER_API_BASE_URL` overridden with empty string

**Root cause:** The field was declared twice in the same class. Pydantic uses the last declaration, so the correct URL was overwritten with `""`.

**Impact:** All weather API requests were sent to an empty URL, causing a network error.
# BEFORE (broken)
class Config(BaseSettings):
    WEATHER_API_BASE_URL: str = "https://api.open-meteo.com/v1/forecast"  # correct
    # ... other fields ...
    WEATHER_API_BASE_URL: str = ""   # duplicate — overrides the real URL!
# AFTER (fixed) — duplicate removed
class Config(BaseSettings):
    WEATHER_API_BASE_URL: str = "https://api.open-meteo.com/v1/forecast"
### Bug 8: `TEMP_MIN` was a string `"0"` used as a temperature label

**Root cause:** `TEMP_MIN` was typed as `str` with value `"0"`, which is not a meaningful temperature classification label.

**Impact:** Temperatures below 0°C would display as `(0)` instead of a readable label like `(freezing)`.
# BEFORE (broken)
TEMP_MIN: str = "0"
# AFTER (fixed)
TEMP_MIN: str = "freezing"
---
## `helper_functions.py` — 1 Bug
### Bug 9: `classify_temperature` first condition caught all non-zero temperatures

**Root cause:** `if temp_celsius:` evaluates to `True` for any non-zero float, so the function returned `config.TEMP_MIN` for virtually every real-world temperature and never reached the other branches.

**Impact:** Every temperature was classified as `"0"` (the broken `TEMP_MIN` value) regardless of the actual value.
# BEFORE (broken)
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius:            # True for ANY non-zero value — always returns here
        return config.TEMP_MIN
    elif temp_celsius < config.TEMP_COLD:
        return "cold"
    elif temp_celsius < config.TEMP_COOL:
        return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE:
        return "comfortable"
    elif temp_celsius < config.TEMP_WARM:
        return "warm"
    else:
        return "hot"
# AFTER (fixed)
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius < 0:        # only matches sub-zero temperatures
        return config.TEMP_MIN  # returns "freezing"
    elif temp_celsius < config.TEMP_COLD:
        return "cold"
    elif temp_celsius < config.TEMP_COOL:
        return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE:
        return "comfortable"
    elif temp_celsius < config.TEMP_WARM:
        return "warm"
    else:
        return "hot"
---
## Verification

After all fixes were applied, the agent ran successfully and produced the following output:

```
============================================================
WEATHER INFORMATION
============================================================
Time: 20:30 UTC | 16:30 (UTC-0400)

Good afternoon, Sukanya!

Your current location: Fairburn, Georgia, United States

Current weather conditions:
• Partly cloudy
• Temperature: 29.1°C (warm)
• Wind: 4.0 km/h
```

> **Note:** The greeting shows "Good afternoon" instead of "Good morning" at 16:30 local time. A further improvement was made to refine `get_greeting()` in `helper_functions.py` to distinguish morning / afternoon / evening based on the actual local hour.